In [ ]:
import threading
import queue

from arraylake import Client
import icechunk
import datetime
import numpy as np
import zarr

from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state
import torch

from main import fetch_initial_conditions, get_gpu_regridder, state_to_xarray, datetime_to_str

In [ ]:
client = Client()
client.login()

In [ ]:
repo = client.get_repo("earthmover-public/aifs-initial-conditions")
session = repo.writable_session("main")
session

In [ ]:
date = datetime.datetime(2025, 6, 25, 0, 0, 0, tzinfo=datetime.UTC)
print("loading initial conditions for", date)
%time fields = fetch_initial_conditions(date, session)

In [ ]:
print("setting up regridder")
regridder = get_gpu_regridder({"grid": "N320"}, {"grid": (0.25, 0.25)})

In [ ]:
checkpoint = {"huggingface": "ecmwf/aifs-single-1.0"}
runner = SimpleRunner(checkpoint, device="cuda")

In [ ]:
target_repo = client.get_or_create_repo("earthmover-demos/private-aifs-forecast")
target_session = target_repo.writable_session("main")

In [ ]:
date_no_tz = date.replace(tzinfo=None)
input_state = dict(date=date_no_tz, fields=fields)

# we put data that we want to write into a queue
q = queue.Queue()
lock = threading.Lock()

def worker():
    while True:
        (ds, store, group_name, kwargs) = q.get()
        # lock is probably unncessary
        with lock:
            ds.to_zarr(
                store, group=group_name, zarr_format=3, consolidated=False, **kwargs
            )
        q.task_done()

# a separate thread for I/O to avoid blocking the main loop
threading.Thread(target=worker, daemon=True).start()

print("starting forecast loop")
kwargs = {"mode": "w"}
# main forecast loop

# clear GPU memory
torch.cuda.empty_cache()

for n, state in enumerate(runner.run(input_state=input_state, lead_time=96)):
    print_state(state)
    ds = state_to_xarray(state, regridder=regridder).chunk()
    group = datetime_to_str(date)
    if n > 0:
        kwargs = {"mode": "a", "append_dim": "valid_time"}
    q.put((ds, target_session.store, group, kwargs))

q.join()  # wait for all I/O tasks to finish

# clear GPU memory
torch.cuda.empty_cache()


In [ ]:
target_session.commit("Wrote a 96 hour forecast")